<a href="https://colab.research.google.com/github/WMFong0/Python-Weather-Report-System/blob/Second-Version-Hotfix/Python_Weather_App.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
#utility.py

def twentyfourh_to_12h(time: str) -> str:
  time = time.split(':')
  hour = int(time[0])
  if hour >= 12:
    time.append("PM")
    if (hour > 12):
      time[0] = str(hour - 12)
  else:
    time.append("AM")
    if (hour == 0):
      time[0] = str(12)

  return ":".join(time[:-1]) + " " + time[-1]

def No_empty_string(received_data):
  if received_data == "":
    return None
  else: return received_data

def select_place(original_selection, raw_data_temperature_datasection: list, variable_name_for_station: str, supported_station: dict = {}):
    # Predefined
    if original_selection in supported_station.keys():
      return District_reference_for_temperature[original_selection]

    available_station = []
    # Fetch available station
    for station_data in raw_data_temperature_datasection:
      available_station.append( station_data[variable_name_for_station] )

    # Remove supported station
    for station in supported_station.values():
      available_station.remove(station)

    # Same Name Case
    if original_selection in available_station:
      return original_selection


    # Required Manual Case
    print("Since we currently doesn't support automatic selection of weather station at your district\n Please select the nearest weather station from the list below: ")
    station = ""

    while True:
      print("Available Station: ")
      for i in range(len(available_station)):
        if (i != 0 and i % 4 == 0):
          print("\t" + available_station[i], end = '\n')
        else:
          print("\t" + available_station[i], end = '')

      station = input("\nKey in your nearest Weather Station: ")
      # Just in case operation
      if (station in available_station):
        print()
        break
      print("You have input a unavailable station. Please input a available Station." + "\n"*2)

    return station

def lazy_list_message(message_list: list = []):
  for message in message_list:
    print(f"\t{message}")
print(twentyfourh_to_12h("18:00:00"))

6:00:00 PM


In [2]:
# cwr.py
  # uvindex, temp, humid

District_reference_for_temperature = {
    'Tin Shui Wai': 'Lau Fau Shan',
    'Yuen Long': 'Yuen Long Park'
}
import requests
import json

class Current_Weather_Report():
  raw_data = None
  temperature = None
  last_update = None
  district = None

  def __init__(self, district: str) -> None:
    self.raw_data = None
    self.temperature = None
    self.last_update = None
    self.processed_data = None
    self.district = district

  def fetch_data(self) -> bool:
    try:
      response = requests.get('https://data.weather.gov.hk/weatherAPI/opendata/weather.php?dataType=rhrread&lang=en', timeout = 10)
      response.raise_for_status()

      if not response:
        raise Exception("Received empty data from API")
        return False

      self.raw_data = response.json()

      if not self.raw_data['updateTime']:
        raise Exception("No updateTime. Possible update on Hong Kong Observatory")
        return False

      self.last_update = self.raw_data['updateTime'][11:19]
      return True

    except Exception as e:
      raise Exception(f"Error getting Current Weather Report data: {str(e)}")

  def fetch_lighting(self) -> dict:

    raw_data_lighting = self.raw_data.get('lightning')

    if (not raw_data_lighting) or raw_data_lighting == "":
      print("Lightning is not happening in Hong Kong right now.\n")
      return

    print(f"Lightning has began at these location,\n"+
          f"starting from {twentyfourh_to_12h(raw_data_lighting['startTime'][11:19])} "+
          f"to {twentyfourh_to_12h(raw_data_lighting['endTime'][11:19])}")

    for temp in raw_data_lighting['data']:
      if temp["occur"] == "true":
        print(f"\t{temp['place']}")

    print()

    return raw_data_lighting

  def fetch_uv(self) -> dict:
    try:
      raw_data_uvindex = self.raw_data.get('uvindex')

      if (not raw_data_uvindex) or raw_data_uvindex == "":
        print("UV Index is unavailable at night.\n")
        return None

      raw_data_uvindex = raw_data_uvindex['data'][0] #Only 1 place available for presenting uvindex data

      raw_data_uvindex['recordDesc'] = self.raw_data['uvindex']['recordDesc']
      raw_data_uvindex['updateTime'] = self.raw_data['updateTime'][11:19]

    except Exception as e:
      raise Exception(f"Error getting uvindex: {str(e)}")

    print(f"\n{raw_data_uvindex['recordDesc']} of {twentyfourh_to_12h(raw_data_uvindex['updateTime'])},",
          f"the current uvindex recorded at {raw_data_uvindex['place']} is {raw_data_uvindex['value']}, " +
          f"classified as {raw_data_uvindex['desc']}")


    extra_message = raw_data_uvindex.get("message")
    if extra_message != None and extra_message != "":
      print(f"Here is a special announcement from HKO regarding uv index: \n",
            extra_message)

    print()

    return raw_data_uvindex

  def fetch_temperature(self) -> dict:
    try:
      raw_data_temperature = self.raw_data.get('temperature')

      if (not raw_data_temperature) or raw_data_temperature == "":
        raise Exception(f"Received Null value in temperature. \n"
        "Report this error to the author.")
        return None

      # print(raw_data_temperature)
      result = {
          'place': select_place(self.district, raw_data_temperature['data'], 'place', District_reference_for_temperature),
          'recordTime': raw_data_temperature['recordTime'][11:19]
      }

      for x in raw_data_temperature['data']:
        if x['place'] == result['place']:

          result['value'] = str(x['value']) + " "+ x['unit']

          print(f"At {twentyfourh_to_12h(result['recordTime'])}, in {result['place']},",
          f"the temperature is {result['value']}\n")
          return result

      raise Exception(f"Data not available at requested district")

    except Exception as e:
      raise Exception(f"Error getting uvindex: {str(e)}")

  def fetch_humidity(self) -> dict:
    raw_data_humidity = self.raw_data.get("humidity")
    if raw_data_humidity == None:
      print("Humidity data unavailable")
      return

    temp = raw_data_humidity.get('data')
    if temp == None or temp == []:
      print("Humidity data unavailable. Possible full mainatance of HKO")
      return
    temp = temp[0]

    result = {
        'place': temp['place'],
        'value': str(temp['value']) + " " + temp['unit'],
        'recordTime': raw_data_humidity['recordTime'][11:19]
    }

    print(f"Humidity data recorded at {twentyfourh_to_12h(result['recordTime'])}: \n" +
          f"\tAt {result['place']}, the humudity recorded is {result['value']}.")

    if (temp['unit'] == 'percent'):
      print(f"\t Humidity Level is ", end = "")
      if (temp['value'] < 25):
        print('Low')
      elif (temp['value'] < 75):
        print('Moderate')
      else:
        print('High')

    return result

  def fetch_and_process_warning(self) -> list:
    result = self.raw_data['warningMessage']

    if result == "":
      return None

    print("\nHere "+
          f"{'is 1' if len(result) == 1 else f'are {len(result)}'} warning message"+
          f"{'s' if len(result)!= 1 else ''} from Hong Kong Observatory:")
    for message in result:
      print(f"\t{message}")
    return result

  def fetch_and_process_ultil(self) -> dict:
    # Special Weather Tips
    specialWxTips = self.raw_data.get('specialWxTips')
    if specialWxTips != None:
      print("\nHere are some special Weather Tips from HKO: ")
      for message in specialWxTips:
        print(f"\t{message}")

    # Tropical Cyclone Position
    tcmessage = self.raw_data.get('tcmessage')
    if (tcmessage != ""):
      print("\nHere are some information relating to Tropical Cyclone from HKO: \n"+
            f"Currently there {'is 1' if len(tcmessage) == 1 else f'are {len(tcmessage)}'} tropical cyclone" +
            f"{'s' if len(tcmessage) != 1 else ''}"+
            "near Hong Kong")
      for message in tcmessage:
        print(f"\t{message}")

    # Others are always unavailable without any reason why
    return None

  def fetch_bundle(self) -> bool:
    try:
      self.fetch_data()
      # print(self.raw_data)
      self.fetch_lighting()
      self.fetch_uv()
      self.fetch_temperature()
      self.fetch_humidity()
      self.fetch_and_process_warning()
      self.fetch_and_process_ultil()
    except Exception as e:
      raise Exception(f"Error fetching data: {str(e)}")

# Quick run Current Weather Report
def run_cwr(district: str):
  cwr = Current_Weather_Report(district)
  cwr.fetch_bundle()

In [3]:
# Hourly_Rainfall.py
import requests
import json

PRIORITY_STATIONS = {'Tuen Mun': ["RF019", "RF002", "RF001" , "N12"], #屯門，#濕地公園，流浮山，元朗水邊圍
                     'Tin Shui Wai': ["RF002", "RF001", "N12", "RF019"], #濕地公園，流浮山，元朗水邊圍，屯門
                     'Yuen Long': ["N12", "RF002","RF003", "RF001"]} #元朗水邊圍, 濕地公園，石崗，流浮山

# A way to check if the station is still close to the district user mentioend (within 3.0 km)
PRIORITY_STATIONS_bool = {
    'Tuen Mun': [True, False, False, False],
    'Tin Shui Wai': [True, True, True, False],
    'Yuen Long': [True, False, False, False]
}

class Hourly_Rainfall():
  raw_data = None
  processed_data = None
  last_update = None
  district = None
  # Manual approach identifier
  manual = False

  def __init__(self, district: str) -> None:
    self.raw_data = None
    self.processed_data = None
    self.last_update = None
    self.district = district
    self.manual = False

  def fetch_data(self) -> bool:
    try:
      response = requests.get('https://data.weather.gov.hk/weatherAPI/opendata/hourlyRainfall.php?lang=en', timeout = 10)
      response.raise_for_status() # For HTTP error

      if not response:
        raise Exception("Received empty data from API")
        return False

      self.raw_data = response.json()

      if not self.raw_data['obsTime']:
        raise Exception("No obsTime. Possible update on Hong Kong Observatory")
        return False

      self.last_update = self.raw_data['obsTime'][11:19]
      return True

    except Exception as e:
      raise Exception(f"Error getting hourlyRainfall data: {str(e)}")

  # Extract data for all priority stations
  def filter_data(self) -> list:
    # Result = [None, None, None, None] normally
    result = [None for i in range(len(PRIORITY_STATIONS[self.district]))]
    try:

      # If hourlyRainfall doesn't exist on raw_data (the data from API request, raise an exception)
      if not (self.raw_data['hourlyRainfall']):
        raise Exception(f"Possible Full Maintenance on Hong Kong Observatory. ")

        return None

      hourlyRainfall_data = self.raw_data['hourlyRainfall']

      # Manual apporach
      if (self.district not in PRIORITY_STATIONS.keys()):
        self.manual = True
        while True:
          self.district = select_place(self.district, hourlyRainfall_data, 'automaticWeatherStation', [])
          for data in hourlyRainfall_data:
            if (data['automaticWeatherStation'] == self.district):
              if (data['value'] != 'M'):
                print(f"During last hour of {self.last_update}, in {data[0]}, the rainfall amount is {data[1]}\n")
                return
              else:
                print("This weather station is under maintenance. Please select another station")
                break

        return None

      for data in hourlyRainfall_data:
        if ((data['automaticWeatherStationID'] in PRIORITY_STATIONS[self.district]) and data['value'] != 'M'):
          index = PRIORITY_STATIONS[self.district].index(data['automaticWeatherStationID']) # as index will raise exception if it doesn't find its answer, very dumb feature
          result[index] = [data['automaticWeatherStation'] , data['value'] + " " + data['unit']]

      if (result != [None, None, None, None]):
        self.processed_data = result
        return result
      else: return None # return none if all 4 of them is not working


    except Exception as e:
      raise Exception(f"Error filtering hourlyRainfall data: {str(e)}")

  def get_nearest_rainfall_data(self):

    for i in range(len(self.processed_data)):
      nearest_rainfall_data = self.processed_data[i]
      if (nearest_rainfall_data):

        if (PRIORITY_STATIONS_bool[self.district][i] == False):
          print("As all the automatic weather station next to you is under maintenance, we will present data from other weather station.\n"
          + "The data retrived might be less accurate. We are sorry for such inconvenience.")

        print(f"During last hour of {self.last_update}, in {nearest_rainfall_data[0]}, the rainfall amount is {nearest_rainfall_data[1]}\n")
        return

    print(f"All 4 nearest automatic weather staion is not available/ under maintenance. Please try again later.") # This shouldn't be used normally. Just a redundant command for just incase.
    return

def run_Hourly_Rainfall(district: str):

  rainfall = Hourly_Rainfall(district)
  try:
      if not (rainfall.fetch_data()):
        raise Exception("Hong Kong Observary API is currently unavailable")

      if (rainfall.filter_data() == None):
        raise Exception("All 4 nearest automatic weather staion is not available/ under maintenance. Please try again later.")

      if (rainfall.manual == False):
        rainfall.get_nearest_rainfall_data()

  except Exception as e:
      print(f"Error: {str(e)}")


In [4]:
# Main.py
import requests
import json
import datetime

current_datetime = datetime.datetime.now().astimezone(datetime.timezone(datetime.timedelta(hours=8))); # Enforce Hong Kong Timezone

print("Welcome using Weather Report System." + "\n" +
      "This system uses Hong Kong Observatory Data to report")

district = input("Please Enter your district: ")

print("\n" + "="*(20+len(f" Weather Report in {district} ")+20) + "\n" + "="*20 + f" Weather Report in {district} " + "="*20 + "\n")

print(current_datetime.strftime('Today is %Y/%m/%d. \nCurrent Time: %H:%M:%S') + "\n")

run_Hourly_Rainfall(district)
run_cwr(district)

Welcome using Weather Report System.
This system uses Hong Kong Observatory Data to report
Please Enter your district: Tuen Mun

==================== Weather Report in Tuen Mun ====================

Today is 2025/08/14. 
Current Time: 23:10:58

During last hour of 22:45:00, in Tuen Mun, the rainfall amount is 0 mm

Lightning has began at these location,
starting from 9:45:00 PM to 10:45:00 PM
[{'place': 'New Territories East', 'occur': 'true'}, {'place': 'Hong Kong and Kowloon', 'occur': 'true'}]
	New Territories East
	Hong Kong and Kowloon

UV Index is unavailable at night.

At 11:00:00 PM, in Tuen Mun, the temperature is 26 C

Humidity data recorded at 11:00:00 PM: 
	At Hong Kong Observatory, the humudity recorded is 92 percent.
	 Humidity Level is High

Here are 2 warning messages from Hong Kong Observatory:
	The Thunderstorm Warning has been issued. It will remain effective until 12:30 a.m. tomorrow. Squally thunderstorms are expected to occur over Hong Kong.
	The Landslip Warning 